In [ ]:
# ── Imports and setup ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score
import shap
import warnings
warnings.filterwarnings('ignore')
import os

OUT  = os.path.join(os.path.dirname(os.path.abspath("__file__")), "outputs")
os.makedirs(OUT, exist_ok=True)

print("All imports OK")
print(f"OUT path exists: {os.path.exists(OUT)}")

In [ ]:
# ──Load Data ──
delta   = pd.read_csv(os.path.join(OUT, "delta_otu_clr.csv"), index_col=0)
meta    = pd.read_csv(os.path.join(OUT, "merged_metadata.csv"))
clr_raw = pd.read_csv(os.path.join(OUT, "preprocessed_otu_clr.csv"), index_col=0)
shap_df = pd.read_csv(os.path.join(OUT, "delta_feature_importances.csv"))

print(f"Delta matrix   : {delta.shape}")
print(f"Metadata       : {meta.shape}")
print(f"CLR (raw)      : {clr_raw.shape}")
print(f"SHAP features  : {shap_df.shape}")


In [ ]:
# ── Extract metadata from delta matrix ──

delta_meta = delta[['subject_id', 'treatment']].copy()
delta = delta.drop(columns=['subject_id', 'treatment'])

# Set index to subject_id
delta.index = delta_meta['subject_id'].values
delta_meta.index = delta_meta['subject_id'].values

print(f"Delta matrix (clean) : {delta.shape}")
print(f"Delta index sample   : {delta.index[:5].tolist()}")
print(f"Treatment counts     : {delta_meta['treatment'].value_counts().to_dict()}")
print(f"Any duplicate index  : {delta.index.duplicated().sum()}")

In [ ]:
# ── Per-subject metadata ──
# Use after-timepoint rows as representative
meta_after = meta[meta['timepoint'] == 'after'].copy()

# Check if subject_id is unique after filtering
print(f"After-timepoint rows     : {len(meta_after)}")
print(f"Unique subject_ids       : {meta_after['subject_id'].nunique()}")
print(f"Duplicate subject_ids    : {meta_after['subject_id'].duplicated().sum()}")

# If duplicates exist, keep first occurrence per subject
meta_sub = (meta_after
            .drop_duplicates(subset='subject_id')
            [['subject_id','treatment','study','fiber_type']]
            .set_index('subject_id'))

print(f"\nMeta_sub shape           : {meta_sub.shape}")
print(f"Treatment counts:\n{meta_sub['treatment'].value_counts()}")

# Align delta to meta_sub — use first occurrence for duplicated subjects
delta_nodup = delta[~delta.index.duplicated(keep='first')]

common_subj = delta_nodup.index.intersection(meta_sub.index)
delta_aligned = delta_nodup.loc[common_subj]
meta_aligned  = meta_sub.loc[common_subj]

print(f"\nCommon subjects          : {len(common_subj)}")
print(f"Delta aligned shape      : {delta_aligned.shape}")
print(f"Meta aligned shape       : {meta_aligned.shape}")

In [ ]:
# ── Shift magnitude ──
shift_mag = np.linalg.norm(delta_aligned.values, axis=1)
meta_aligned = meta_aligned.copy()
meta_aligned['shift_magnitude'] = shift_mag

# Fiber arm only
fiber_meta = meta_aligned[meta_aligned['treatment'] == 'fiber'].copy()

print(f"Fiber arm subjects       : {len(fiber_meta)}")
print(f"Shift magnitude stats:")
print(f"  Min    : {fiber_meta['shift_magnitude'].min():.3f}")
print(f"  Median : {fiber_meta['shift_magnitude'].median():.3f}")
print(f"  Max    : {fiber_meta['shift_magnitude'].max():.3f}")


In [ ]:
# ── Study-stratified responder definition ──

def assign_responder_stratified(group):
    t33 = np.percentile(group['shift_magnitude'], 33.33)
    t67 = np.percentile(group['shift_magnitude'], 66.67)
    def label(x):
        if x >= t67:   return 'responder'
        elif x <= t33: return 'non_responder'
        else:          return 'middle'
    group = group.copy()
    group['responder_label'] = group['shift_magnitude'].apply(label)
    group['t33'] = t33
    group['t67'] = t67
    return group

fiber_meta = fiber_meta.groupby('study', group_keys=False).apply(
    assign_responder_stratified
)

print(f"Responder label counts (study-stratified):")
print(fiber_meta['responder_label'].value_counts())

print(f"\nBy study:")
print(fiber_meta.groupby('study')['responder_label'].value_counts().unstack(fill_value=0))

# Overwrite saved file with corrected labels
fiber_meta.to_csv(os.path.join(OUT, "delta_otu_clr.csv"))
print("\nSaved: responder_labels.csv (study-stratified)")

In [ ]:
# ── Figure: shift magnitude distribution ──

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: Violin + strip colored by responder label
ax = axes[0]
palette = {'responder': '#e04b3a', 'non_responder': '#4b79b5', 'middle': '#bbbbbb'}

sns.violinplot(data=fiber_meta, y='shift_magnitude', ax=ax,
               color='#eeeeee', inner=None, linewidth=1.2)
sns.stripplot(data=fiber_meta, y='shift_magnitude',
              hue='responder_label', palette=palette,
              ax=ax, size=3.5, alpha=0.7, jitter=True, legend=False)

# Per-study tertile lines (show spread)
for study, grp in fiber_meta.groupby('study'):
    ax.axhline(grp['t67'].iloc[0], color='#e04b3a', alpha=0.15, linewidth=0.8)
    ax.axhline(grp['t33'].iloc[0], color='#4b79b5', alpha=0.15, linewidth=0.8)

handles = [mpatches.Patch(color=v, label=k) for k, v in palette.items()]
ax.legend(handles=handles, fontsize=8)
ax.set_title('Microbiome Shift Magnitude\n(study-stratified tertiles)', fontsize=11)
ax.set_ylabel('Euclidean distance in CLR space', fontsize=10)
ax.set_xlabel('')

# Panel B: Boxplot by study, colored by median shift
ax2 = axes[1]
study_order = (fiber_meta.groupby('study')['shift_magnitude']
               .median().sort_values(ascending=False).index.tolist())
sns.boxplot(data=fiber_meta, x='study', y='shift_magnitude',
            order=study_order, ax=ax2, color='#ddeeff',
            flierprops=dict(marker='o', markersize=3, alpha=0.5))
ax2.set_xticklabels([s.replace('_', '\n') for s in study_order], fontsize=7)
ax2.set_title('Shift Magnitude by Study', fontsize=11)
ax2.set_ylabel('Euclidean distance in CLR space', fontsize=10)
ax2.set_xlabel('')

plt.tight_layout()
plt.savefig(os.path.join(OUT, "delta_otu_clr.csv"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_responder_shift_dist.png")

In [ ]:
# ── Responder classifier (LOSO-CV) ──

# Drop middle tertile
resp_mask = fiber_meta['responder_label'].isin(['responder', 'non_responder'])
resp_meta = fiber_meta[resp_mask].copy()

X_resp = delta_aligned.loc[resp_meta.index]
y_resp = (resp_meta['responder_label'] == 'responder').astype(int).values
groups = resp_meta['study'].values

print(f"Classifier input:")
print(f"  Subjects     : {len(y_resp)}  (responders={y_resp.sum()}, non={len(y_resp)-y_resp.sum()})")
print(f"  Features     : {X_resp.shape[1]}")
print(f"  Studies      : {np.unique(groups).tolist()}")
print(f"  Class balance: {y_resp.mean():.3f}")

logo = LeaveOneGroupOut()
results = []

for train_idx, test_idx in logo.split(X_resp, y_resp, groups):
    study_name = np.unique(groups[test_idx])[0]
    if len(np.unique(y_resp[test_idx])) < 2:
        print(f"  Skipping {study_name} — single class in test set")
        continue

    clf = RandomForestClassifier(
        n_estimators=500, max_features=0.1,
        min_samples_leaf=1, class_weight='balanced',
        random_state=SEED, n_jobs=-1
    )
    clf.fit(X_resp.iloc[train_idx], y_resp[train_idx])
    proba = clf.predict_proba(X_resp.iloc[test_idx])[:, 1]
    auc   = roc_auc_score(y_resp[test_idx], proba)

    results.append({
        'study'        : study_name,
        'n_test'       : len(test_idx),
        'n_responder'  : int(y_resp[test_idx].sum()),
        'n_non_responder': int(len(test_idx) - y_resp[test_idx].sum()),
        'AUC'          : round(auc, 4)
    })
    print(f"  {study_name:<25} AUC = {auc:.4f}  (n={len(test_idx)})")

resp_df = pd.DataFrame(results)
print(f"\nMean AUC : {resp_df['AUC'].mean():.4f} ± {resp_df['AUC'].std():.4f}")
resp_df.to_csv(os.path.join(OUT, "delta_otu_clr.csv"), index=False)
print("Saved: responder_rf_results.csv")

In [ ]:
# ── Responder classifier summary ──
print(f"\nResponder vs Non-Responder Classifier — LOSO-CV Results")
print(f"{'Study':<25} {'N':>4} {'AUC':>6}")
print("-" * 38)
for _, row in resp_df.iterrows():
    print(f"  {row['study']:<23} {int(row['n_test']):>4} {row['AUC']:>6.4f}")
print("-" * 38)
print(f"  {'Mean ± SD':<23}      {resp_df['AUC'].mean():.4f} ± {resp_df['AUC'].std():.4f}")
print(f"  {'Median':<23}      {resp_df['AUC'].median():.4f}")
print(f"\nComparison to original delta CLR classifier:")
print(f"  Original (fiber vs control) : 0.624 ± 0.136")
print(f"  Responder classifier        : {resp_df['AUC'].mean():.3f} ± {resp_df['AUC'].std():.3f}")
print(f"  Improvement                 : +{resp_df['AUC'].mean() - 0.624:.3f} AUC points")

In [ ]:
# ── Baseline sample ID alignment ──

# Add study column to before_map
before_map2 = (meta_before
               .drop_duplicates(subset='subject_id')
               [['subject_id', 'sample_id_2', 'study']]
               .set_index('subject_id'))

# Restrict to responder subjects
common_baseline = resp_meta.index.intersection(before_map2.index)
before_info     = before_map2.loc[common_baseline].copy()

# Build prefixed sample ID: study + "_" + sample_id_2
before_info['sample_id_prefixed'] = (before_info['study'] + '_' + 
                                      before_info['sample_id_2'])

# Check join
valid_mask = before_info['sample_id_prefixed'].isin(clr_raw.index)
print(f"Responder subjects with before-map  : {len(before_info)}")
print(f"Sample IDs found in CLR matrix      : {valid_mask.sum()} / {len(valid_mask)}")

if (~valid_mask).sum() > 0:
    print(f"\nStill missing (first 5):")
    print(before_info.loc[~valid_mask, 'sample_id_prefixed'].head().tolist())

In [ ]:
# ── Baseline predictor matrix ──
before_info_valid = before_info[valid_mask].copy()

X_base  = clr_raw.loc[before_info_valid['sample_id_prefixed'].values].copy()
X_base.index = before_info_valid.index  # reindex to subject_id

y_base   = (resp_meta.loc[before_info_valid.index, 'responder_label'] == 'responder').astype(int).values
grp_base = resp_meta.loc[before_info_valid.index, 'study'].values

print(f"Baseline predictor input:")
print(f"  Subjects     : {len(y_base)}  (responders={y_base.sum()}, non={len(y_base)-y_base.sum()})")
print(f"  Features     : {X_base.shape[1]}")
print(f"  Class balance: {y_base.mean():.3f}")
print(f"\nBy study:")
for s in np.unique(grp_base):
    mask = grp_base == s
    print(f"  {s:<25} n={mask.sum()}  resp={y_base[mask].sum()}  non={mask.sum()-y_base[mask].sum()}")

In [ ]:
# ── Baseline predictor LOSO-CV ──
base_results = []

for train_idx, test_idx in logo.split(X_base, y_base, grp_base):
    study_name = np.unique(grp_base[test_idx])[0]
    if len(np.unique(y_base[test_idx])) < 2:
        print(f"  Skipping {study_name} — single class in test set")
        continue

    clf = RandomForestClassifier(
        n_estimators=500, max_features=0.1,
        min_samples_leaf=1, class_weight='balanced',
        random_state=SEED, n_jobs=-1
    )
    clf.fit(X_base.iloc[train_idx], y_base[train_idx])
    proba = clf.predict_proba(X_base.iloc[test_idx])[:, 1]
    auc   = roc_auc_score(y_base[test_idx], proba)

    base_results.append({
        'study'  : study_name,
        'n_test' : len(test_idx),
        'n_responder': int(y_base[test_idx].sum()),
        'n_non_responder': int(len(test_idx) - y_base[test_idx].sum()),
        'AUC'    : round(auc, 4)
    })
    print(f"  {study_name:<25} AUC = {auc:.4f}  (n={len(test_idx)})")

base_df = pd.DataFrame(base_results)
print(f"\n{'Study':<25} {'N':>4} {'AUC':>6}")
print("-" * 38)
for _, row in base_df.iterrows():
    print(f"  {row['study']:<23} {int(row['n_test']):>4} {row['AUC']:>6.4f}")
print("-" * 38)
print(f"  {'Mean ± SD':<23}      {base_df['AUC'].mean():.4f} ± {base_df['AUC'].std():.4f}")
print(f"  {'Median':<23}      {base_df['AUC'].median():.4f}")

base_df.to_csv(os.path.join(OUT, "delta_otu_clr.csv"), index=False)
print("\nSaved: baseline_predictor_results.csv")

In [ ]:
# ── Global model and SHAP values ──
print("Training global model...")
clf_global = RandomForestClassifier(
    n_estimators=500, max_features=0.1,
    min_samples_leaf=1, class_weight='balanced',
    random_state=SEED, n_jobs=-1
)
clf_global.fit(X_base, y_base)
print("Model trained.")

print("Computing SHAP values...")
explainer   = shap.TreeExplainer(clf_global)
shap_values = explainer.shap_values(X_base)

# Debug output format
print(f"\nType of shap_values     : {type(shap_values)}")
if isinstance(shap_values, list):
    print(f"List length             : {len(shap_values)}")
    for i, sv in enumerate(shap_values):
        print(f"  shap_values[{i}] shape  : {np.array(sv).shape}")
else:
    print(f"Array shape             : {np.array(shap_values).shape}")

In [ ]:
# ── Extract SHAP values (class 1) ──
# shap_values shape: (236, 9612, 2) — take class 1 (responder)
sv = shap_values[:, :, 1]
print(f"SHAP values for class 1 (responder) shape: {sv.shape}")

mean_abs_shap = np.abs(sv).mean(axis=0)
print(f"Mean abs SHAP shape: {mean_abs_shap.shape}")

baseline_shap_df = pd.DataFrame({
    'OTU_ID'                : X_base.columns,
    'baseline_mean_abs_shap': mean_abs_shap
}).sort_values('baseline_mean_abs_shap', ascending=False).reset_index(drop=True)
baseline_shap_df['rank'] = baseline_shap_df.index + 1

print("\nTop 20 baseline predictors of fiber response:")
print(f"{'Rank':<6} {'OTU_ID':<45} {'Mean |SHAP|':>12}")
print("-" * 65)
for _, row in baseline_shap_df.head(20).iterrows():
    print(f"  {int(row['rank']):<4} {row['OTU_ID']:<45} {row['baseline_mean_abs_shap']:>10.5f}")

baseline_shap_df.to_csv(os.path.join(OUT, "delta_otu_clr.csv"), index=False)
print("\nSaved: baseline_shap_importances.csv")

In [ ]:
# ── Load taxonomy ──
taxonomy = pd.read_csv(os.path.join(OUT, "delta_otu_clr.csv"), index_col=0)

print(f"Taxonomy table shape : {taxonomy.shape}")
print(f"Taxonomy columns     : {taxonomy.columns.tolist()}")
print(f"Taxonomy index sample: {taxonomy.index[:3].tolist()}")


In [ ]:
# ── Annotate baseline SHAP top OTUs ──
# Taxonomy strings are GTDB format: d__;p__;c__;o__;f__;g__;s__
# Split into columns

tax_split = taxonomy['taxonomy'].str.split(';', expand=True)
tax_split.columns = ['domain','phylum','class','order','family','genus','species']

# Strip whitespace and GTDB prefixes
for col in tax_split.columns:
    tax_split[col] = tax_split[col].str.strip().str.replace(r'^[dpcofgs]__','',regex=True)

# Join with baseline SHAP rankings
top50_shap = baseline_shap_df.head(50).copy()
top50_annotated = top50_shap.merge(
    tax_split[['genus','species','family','phylum']],
    left_on='OTU_ID', right_index=True, how='left'
)

print("Top 20 baseline predictors — annotated:")
print(f"{'Rank':<5} {'Genus':<25} {'Species':<30} {'Mean |SHAP|':>12}")
print("-" * 75)
for _, row in top50_annotated.head(20).iterrows():
    genus   = str(row['genus'])[:24]   if pd.notna(row['genus'])   else 'Unknown'
    species = str(row['species'])[:29] if pd.notna(row['species']) else ''
    print(f"  {int(row['rank']):<3} {genus:<25} {species:<30} {row['baseline_mean_abs_shap']:>10.5f}")

top50_annotated.to_csv(os.path.join(OUT, "delta_otu_clr.csv"), index=False)
print("\nSaved: baseline_shap_top50_annotated.csv")

In [ ]:
# ── Figure: AUC comparison ──
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
for i, (ov, rv, bv) in enumerate(zip(orig_vals, resp_vals, base_vals)):
    if not np.isnan(ov):
        ax.bar(i - w, ov, w, color='#888888', alpha=0.8)
    if not np.isnan(rv):
        ax.bar(i,     rv, w, color='#e04b3a', alpha=0.8)
    if not np.isnan(bv):
        ax.bar(i + w, bv, w, color='#4b79b5', alpha=0.8)

ax.axhline(0.5, color='black', linestyle='--', linewidth=0.9)
ax.axhline(np.mean(valid_orig),     color='#888888', linestyle=':', linewidth=1.5)
ax.axhline(resp_df['AUC'].mean(),   color='#e04b3a', linestyle=':', linewidth=1.5)
ax.axhline(base_df['AUC'].mean(),   color='#4b79b5', linestyle=':', linewidth=1.5)

ax.set_xticks(x)
ax.set_xticklabels(all_studies, fontsize=8, rotation=90)
ax.set_ylim(0.4, 1.05)
ax.set_ylabel('AUC (LOSO-CV)', fontsize=10)
ax.set_title('Classifier Performance Comparison', fontsize=11)
ax.legend(handles=legend_elements, fontsize=8,
          bbox_to_anchor=(0.5, -0.55), loc='upper center',
          ncol=1, frameon=True)

# Panel B unchanged
ax2 = axes[1]
ax2.barh(range(15), top15['baseline_mean_abs_shap'].values[::-1],
         color=colors_bar[::-1], alpha=0.8)
ax2.set_yticks(range(15))
ax2.set_yticklabels(clean_labels[::-1], fontsize=8)
ax2.set_xlabel('Mean |SHAP value| (baseline CLR)', fontsize=9)
ax2.set_title('Top 15 Baseline Predictors of Fiber Response\n(red = known SCFA producers)', fontsize=10)

plt.tight_layout()
plt.subplots_adjust(bottom=0.32)
plt.savefig(os.path.join(OUT, "delta_otu_clr.csv"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_responder_auc.png")

In [ ]:
# ── Summary ──
print("=" * 60)
print("  RESPONDER ANALYSIS — COMPLETE SUMMARY")
print("=" * 60)

print(f"""
DATASET
  Fiber arm subjects          : {len(fiber_meta)}
  Responders (top tertile)    : {(fiber_meta['responder_label']=='responder').sum()}
  Non-responders (bot tertile): {(fiber_meta['responder_label']=='non_responder').sum()}
  Middle tertile (excluded)   : {(fiber_meta['responder_label']=='middle').sum()}
  Tertile definition          : study-stratified

RESPONDER CLASSIFIER (delta CLR features)
  Mean AUC  : {resp_df['AUC'].mean():.4f} ± {resp_df['AUC'].std():.4f}
  Median AUC: {resp_df['AUC'].median():.4f}
  Min AUC   : {resp_df['AUC'].min():.4f} ({resp_df.loc[resp_df['AUC'].idxmin(),'study']})
  Max AUC   : {resp_df['AUC'].max():.4f} ({resp_df.loc[resp_df['AUC'].idxmax(),'study']})

BASELINE PREDICTOR (pre-intervention CLR features)
  Mean AUC  : {base_df['AUC'].mean():.4f} ± {base_df['AUC'].std():.4f}
  Median AUC: {base_df['AUC'].median():.4f}
  Min AUC   : {base_df['AUC'].min():.4f} ({base_df.loc[base_df['AUC'].idxmin(),'study']})
  Max AUC   : {base_df['AUC'].max():.4f} ({base_df.loc[base_df['AUC'].idxmax(),'study']})

COMPARISON
  Original delta CLR (fiber vs control) : 0.624 ± 0.136
  Responder classifier                  : {resp_df['AUC'].mean():.3f} ± {resp_df['AUC'].std():.3f}
  Baseline predictor                    : {base_df['AUC'].mean():.3f} ± {base_df['AUC'].std():.3f}

TOP 5 BASELINE PREDICTORS
""")
for _, row in top50_annotated.head(5).iterrows():
    print(f"  Rank {int(row['rank'])}: {row['genus']} {row['species']}  "
          f"(SHAP={row['baseline_mean_abs_shap']:.5f})")

print(f"""
OUTPUTS SAVED
  responder_labels.csv
  responder_rf_results.csv
  baseline_predictor_results.csv
  baseline_shap_importances.csv
  baseline_shap_top50_annotated.csv
  fig_responder_shift_dist.png
  fig_responder_auc.png
""")
print("=" * 60)